# lean-agent — showcase

This walks through what the repo does, ending with the question it's built to answer: **does giving the agent well-structured definitions make it prove Lean theorems more effectively?**

Setup (once): `uv sync`, put `OPENAI_API_KEY` in `.env`, and install Lean:

```sh
curl -sSf https://elan.lean-lang.org/elan-init.sh | sh -s -- -y
~/.elan/bin/elan default leanprover/lean4:v4.29.1
```

The next cell puts `~/.elan/bin` on `PATH` for you (Jupyter kernels usually don't inherit your shell `PATH`). The first Lean call builds the REPL (one-time).

In [ ]:
import os

# Jupyter kernels often don't inherit your shell PATH, so put elan's Lean toolchain
# (where `lake` / `lean` live) on PATH before the first Lean call.
elan_bin = os.path.expanduser("~/.elan/bin")
if os.path.isdir(elan_bin) and elan_bin not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = elan_bin + os.pathsep + os.environ["PATH"]

from lean_agent import Lean, lean_config, get_settings

s = get_settings()
print("model:", s.model_id, "| lean:", s.lean_version, "| key set:", bool(s.api_key))

## 1. Give Lean a definition, then prove against it

This is the core mechanism. A **preamble** (imports + definitions/lemmas) is run **once** to build a persistent environment; later proofs run *in* it, so the definitions are in scope. Here, no Mathlib — just core Lean.

In [ ]:
definitions = (
    "def IsEven (n : Nat) : Prop := \u2203 k, n = 2 * k\n"
    "theorem isEven_add_self (n : Nat) : IsEven (n + n) := \u27e8n, by omega\u27e9"
)

lean = Lean(lean_config(definitions))     # a core-Lean REPL session (builds once)
env = lean.base_env(definitions)          # pre-warm: load IsEven + the lemma

# Prove a goal that USES the definition, in that environment:
ok = lean.check("theorem t (n : Nat) : IsEven (n + n) := isEven_add_self n", env)
print("with the definitions loaded :", ok.feedback)

# The SAME proof in a bare environment (no definitions) fails — IsEven is unknown:
bare = lean.check("theorem t (n : Nat) : IsEven (n + n) := isEven_add_self n", None)
print("without them                :", bare.feedback.splitlines()[0])

## 2. Hand the agent a problem — definitions pre-loaded

`solve(problem, lean=...)` pre-warms **both** the Lean environment (the preamble) **and** the model's system prompt (the same preamble), then lets the agent iterate with `lean_check`. Problems come from the `benchmarks/` folder; an *experiment* is a folder of conditions.

In [ ]:
from lean_agent import solve, build_model
from benchmarks import load_experiment

model = build_model()
(notated,) = load_experiment("even_self", names=["notated"])
print("goal:", notated.statement)
print("pre-loaded preamble:\n", notated.preamble, "\n")

result = solve(notated, lean=lean, model=model, max_steps=5)
print(result["passed"], "in", result["steps"], "steps  ->  see", result["run_dir"], "/ transcript.yaml")

## 3. The hypothesis: notated vs raw

`even_self` has two conditions — the same proposition, once with the definition + lemma (`notated`), once desugared with nothing given (`raw`). Run both and compare. (The CLI does this with `uv run python run.py --experiment even_self`.)

In [ ]:
for cond in load_experiment("even_self"):
    r = solve(cond, lean=lean, model=model, max_steps=6)
    print(f"{cond.name:20s} passed={r['passed']}  steps={r['steps']}  out_tokens={r['output_tokens']}")

## 4. Make your own experiment

Drop a folder `benchmarks/data/experiments/<name>/` with one `.lean` file per condition. In each, put your imports + definitions, then the goal as the **last** `theorem ... := sorry` (everything above it is pre-loaded). Then run:

```sh
uv run python run.py --experiment <name>
```

For a real (Mathlib) study, set `LEAN_PROJECT` to a **built** Mathlib project so the conditions compile against it.

Every run is logged under `logs/<run-folder>/`: `run.json` (structured) and **`transcript.yaml`** (the system / user / assistant / tool-call / tool-response lineage, top to bottom) — read that to see exactly what the model saw and did.